# RedZone world model: speedrun recipe A/B at the EXP-103 size (EXP-109), on one Colab A100

Runs BEFORE the 60m/150m/300m ladder: the ladder trains on whatever wins here. Same size as EXP-103 (18m preset: d 384,
8 layers, 6 heads), EXP-103's data (nflverse NFL, NBA.com cdn NBA, Retrosheet MLB; vocab v1) and split, one seed,
**15 min of training per arm**:
A base (AdamW, cosine) | B +Muon | C B+warmup-stable-decay | D C+modded-nanoGPT bundle (QK-norm, ReLU^2, untied head,
soft-cap 15, value embeddings) | E D+torch.compile | F E+doc masking (FlexAttention) | G base+bundle | H base+compile.
Each arm records valid loss at 15 min, valid loss per league, and tokens/hours to reach EXP-103's 0.334.

Runtime > Change runtime type > **A100 GPU** (High-RAM), then **Run all**. About 2.5 h (~32 CU): data build ~25 min
(shared by all arms), 8 x 15 min training, grading skipped (selection only). Finished arms are skipped on a re-run.
The last cell prints the comparison table. Personal-use tag as always.


In [ ]:
PIN = "8b2ea863d93dd9b47abfa137741cd0713971583a"   # kaggle-inputs commit: factory code pack (nfm_train.py with the EXP-109 flags)
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba,mlb"
ARMS = {
    "A_base": [],
    "B_muon": ["--muon"],
    "C_muon_wsd": ["--muon", "--sched", "wsd"],
    "D_bundle": ["--muon", "--sched", "wsd", "--qk-norm", "--relu2", "--untie", "--softcap", "15", "--value-emb"],
    "E_compile": ["--muon", "--sched", "wsd", "--qk-norm", "--relu2", "--untie", "--softcap", "15", "--value-emb",
                  "--compile"],
    "F_docmask": ["--muon", "--sched", "wsd", "--qk-norm", "--relu2", "--untie", "--softcap", "15", "--value-emb",
                  "--compile", "--doc-mask"],
    "G_bundle_only": ["--qk-norm", "--relu2", "--untie", "--softcap", "15", "--value-emb"],
    "H_compile_only": ["--compile"],
}
MINUTES, CTX, BATCH, LR, TARGET = 15, 1024, 64, 6e-4, 0.334     # EXP-103 batch + lr; ctx 1024 (FlexAttention blocks)
TRAIN_TOKENS = 2e9          # never the limit: the 15-min clock is; the WSD decay is set from the steps that fit
RUN, CODE = "/content/rz", "/content/rz/code"
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)


In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
os.makedirs(CODE, exist_ok=True)
if not os.path.exists(f"{CODE}/rz/scripts/nfm_train.py"):
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))


In [ ]:
def run(arm, flags):
    out = f"{RUN}/speedrun/{arm}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/grades.json"):
        print(arm, "done already"); return
    cmd = [sys.executable, "-u", f"{CODE}/rz/scripts/nfm_train.py", "--out", out, "--ckpt-dir", out,
           "--data-dir", f"{RUN}/data", "--build-vocab", "1", "--leagues", LEAGUES, "--preset", "18m",
           "--ctx", str(CTX), "--batch", str(BATCH), "--lr", str(LR), "--warmup", "200",
           "--train-tokens", str(TRAIN_TOKENS), "--minutes", str(MINUTES), "--target-loss", str(TARGET),
           "--throughput-only", "--exp", "EXP-109", *flags]
    if "--sched" in flags:      # WSD needs a known end: the steps that fit in 15 min, measured by arm A
        a = f"{RUN}/speedrun/A_base/grades.json"
        if os.path.exists(a):
            cmd += ["--train-tokens", str(json.load(open(a))["tokens_seen"])]
    with open(f"{out}/train.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in p.stdout:
            print(line, end=""); log.write(line)
        print(arm, "exit code:", p.wait())

for arm, flags in ARMS.items():
    run(arm, flags)


In [ ]:
rows = []
for arm in ARMS:
    gp = f"{RUN}/speedrun/{arm}/grades.json"
    if os.path.exists(gp):
        g = json.load(open(gp))
        t = (g.get("target") or {}).get("reached")
        rows.append((arm, g["best_valid_loss"], g["valid_loss_by_league"], g["tokens_seen"],
                     g["throughput"]["tokens_per_s"], t and t["tokens"], t and t["train_hours"]))
print("arm | best valid | per league | tokens seen | tok/s | tokens to 0.334 | hours to 0.334")
for r in rows:
    print(" | ".join(str(x) for x in r))
json.dump(rows, open(f"{RUN}/speedrun_summary.json", "w"))
print(f"session wall clock: {(time.time() - T_START) / 3600:.2f} h, CU ~{(time.time() - T_START) / 3600 * (RATE or 0):.1f}")


In [ ]:
# EXP-110 (~15 min): FIM-PP zero-shot break timing vs the per-mark median, on the same v1 test nights.
# OpenFIM (MIT) from a pinned commit on PYTHONPATH: its pip package pins python < 3.13 and builds with setuptools_scm,
# which is what broke `pip install git+...` in EXP-109's session. Only its runtime imports are installed.
FIM_COMMIT = "cee2bb53be1e8e92b73f98dfdbe3605efc9119cc"
try:
    fim = f"{RUN}/OpenFIM"
    if not os.path.exists(f"{fim}/src/fim"):
        subprocess.run(["git", "clone", "-q", "https://github.com/FIM4Science/OpenFIM", fim], check=True)
    subprocess.run(["git", "-C", fim, "checkout", "-q", FIM_COMMIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.56.2", "huggingface_hub==0.36.2",
                    "accelerate", "einops", "torchdiffeq", "torchsde", "optree", "pydantic", "h5py", "python-dotenv",
                    "loguru", "torchinfo", "peft", "datasets", "psutil", "gitpython", "scikit-learn", "pyyaml"], check=True)
    env = {**os.environ, "PYTHONPATH": f"{fim}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")}
    for lg in ("nba", "nfl"):          # NBA first; MLB (Retrosheet) has no clock to time breaks with
        out = f"{RUN}/exp110/{lg}"
        p = subprocess.run([sys.executable, "-u", f"{CODE}/rz/scripts/exp110_fimpp.py", "--tokens", f"{RUN}/data/tokens/{lg}",
                            "--leagues", lg, "--out", out, "--max-test", "100"], cwd=RUN, text=True, env=env)
        if os.path.exists(f"{out}/grades.json"):
            print(open(f"{out}/grades.json").read())
        else:
            print(lg, "EXP-110 gave no grades.json (exit", p.returncode, ")")
except Exception as e:
    print("EXP-110 did not run:", type(e).__name__, e, "(the speedrun results above are unaffected)")
